# 05_bronze_ingest

Ingest raw JSONL from GCS into a Bronze Delta table with minimal transformations.

In [ ]:
# Widgets
dbutils.widgets.text("raw_input_path", "gs://liquid-layout-413121-llmfb-raw-dev/raw/llm_feedback_eval/dt=*/run_id=*/batch_id=*/part-00000.jsonl")
dbutils.widgets.text("bronze_output_path", "gs://liquid-layout-413121-llmfb-bronze-dev/bronze/llm_feedback_eval/raw_rows/")
dbutils.widgets.text("bronze_table_name", "bronze.llm_feedback_raw_rows")

raw_input_path = dbutils.widgets.get("raw_input_path").strip()
bronze_output_path = dbutils.widgets.get("bronze_output_path").strip()
bronze_table_name = dbutils.widgets.get("bronze_table_name").strip()

if not raw_input_path or not bronze_output_path or not bronze_table_name:
    raise ValueError("All widget values must be provided.")

In [ ]:
# Fail fast if the GCS path is not accessible
try:
    _ = dbutils.fs.ls(raw_input_path)
except Exception as exc:
    raise RuntimeError(f"Raw input path is not accessible: {raw_input_path}") from exc

In [ ]:
from pyspark.sql import functions as F

df = spark.read.json(raw_input_path)

required_cols = {"meta", "payload"}
missing = required_cols.difference(set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {sorted(list(missing))}")

In [ ]:
df_out = (
    df
    .withColumn("run_id", F.col("meta.run_id"))
    .withColumn("ingest_ts", F.col("meta.ingest_ts"))
    .withColumn("ingest_date", F.to_date(F.to_timestamp(F.col("meta.ingest_ts"))))
    .withColumn("source_name", F.col("meta.source_name"))
    .withColumn("source_type", F.col("meta.source_type"))
    .withColumn("pod_name", F.col("meta.pod_name"))
    .withColumn("pod_type", F.col("meta.pod_type"))
    .withColumn("task_type", F.col("meta.task_type"))
    .withColumn("batch_id", F.coalesce(F.col("payload.batch_id"), F.col("meta.source_file")))
    .withColumn("set_id", F.col("payload.set_id"))
    .withColumn("prompt_id", F.col("payload.prompt_id"))
    .withColumn("step_index", F.col("payload.step_index"))
    .withColumn("evaluated_step_index", F.coalesce(F.col("payload.evaluated_step_index"), F.col("payload.step_index")))
)

In [ ]:
(
    df_out.write
    .format("delta")
    .mode("append")
    .partitionBy("ingest_date", "batch_id")
    .save(bronze_output_path)
)

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql(
    f"CREATE TABLE IF NOT EXISTS {bronze_table_name} USING DELTA LOCATION '{bronze_output_path}'"
)

In [ ]:
total_rows = df_out.count()
distinct_run_ids = df_out.select("run_id").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct run_id: {distinct_run_ids}")

display(
    df_out.groupBy("batch_id").count().orderBy(F.desc("count"))
)

display(
    df_out.select(
        "run_id", "ingest_ts", "batch_id", "set_id", "prompt_id", "step_index", "pod_name", "task_type"
    ).limit(20)
)

## Validation SQL

```sql
SELECT count(*) FROM bronze.llm_feedback_raw_rows;
SELECT batch_id, count(*) FROM bronze.llm_feedback_raw_rows GROUP BY batch_id;
SELECT payload.step_type, count(*) FROM bronze.llm_feedback_raw_rows GROUP BY payload.step_type;
```